# TXF session-flow persistence v0.1.0
Live FinLab data, immutable Google Drive raw snapshots, append-only research results.
**Not a trading signal. Do not confuse net trading contracts with OI change.**

In [ ]:
from pathlib import Path
import subprocess
repo_dir = Path("/content/txf_session_flow_persistence")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/hh4832/txf_session_flow_persistence.git", str(repo_dir)], check=True)
%cd /content/txf_session_flow_persistence
!python -m pip -q install -e ".[finlab,drive]"

## Authenticate
Uses Colab Secrets named `FINLAB_API_TOKEN` if data is not already cached. Do not print the value.
Drive OAuth authentication grants the notebook access to your pre-existing folders.

In [ ]:
import os
from google.colab import auth, userdata
auth.authenticate_user()
try:
    token = userdata.get("FINLAB_API_TOKEN")
    if token:
        os.environ["FINLAB_API_TOKEN"] = token
except Exception:
    print("No token loaded; this is okay only when both raw snapshots already exist.")

## Run research
Set END_DATE to the last exchange trade date that both FinLab feeds have published.
Default column label: 臺股期貨_外資及陸資 — change INSTRUMENT if your FinLab table uses another exact label.
`REFRESH` must be True only to explicitly create a NEW immutable revision; never overwrites earlier snapshots.

In [ ]:
from txf_session_flow_persistence.config import RAW_FOLDER_ID, RESULTS_FOLDER_ID, ResearchConfig
from txf_session_flow_persistence.storage import GoogleDriveStore
from txf_session_flow_persistence.runner import run
START_DATE = "2023-10-16"
END_DATE = "2026-10-22"
INSTRUMENT = "臺股期貨_外資及陸資"
REFRESH = False
cfg = ResearchConfig(start=START_DATE, end=END_DATE, instrument=INSTRUMENT).validate()
manifest, tables = run(GoogleDriveStore(RAW_FOLDER_ID, RESULTS_FOLDER_ID), cfg, refresh=REFRESH)
print({k: manifest[k] for k in ["status", "run_name", "full_trading_dates", "valid_paired_dates", "unusable_dates"]})
display(tables["direction_summary"])